# 6.5940 · Long-context rank follow-up

This CPU replay compares the original and corrected rank baseline at B=96. It also recomputes the reuse-prediction AUC with a tie correction and a complete-window sensitivity check. The second part runs the all-page rank control for the score-aware EMA (z vs all-page rank vs selected-only rank).

You need `rank-followup-colab.zip` (code and archived controls) and the **original 14 `longctx_v1` dev NPZs** in a single Drive folder. These are the long-context traces with `rid`, `ranked`, and `scores`. The earlier top-K-only Colab export is for a different split.

Start with `SMOKE = True` to compare one layer against the archived LRU and Bélády counts. Then switch it to `False` and rerun the configuration and replay cell. Results are written to Drive. A CPU runtime is sufficient.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import io

from google.colab import drive, files

drive.mount('/content/drive')
uploaded = files.upload()
bundle = uploaded.get('rank-followup-colab.zip')
if bundle is None:
    raise ValueError('Upload rank-followup-colab.zip in this cell.')
with ZipFile(io.BytesIO(bundle)) as source:
    source.extractall('/content')
project = Path('/content/rank-followup')
assert (project / 'validation/rerun_longctx_dev.py').is_file()

In [ ]:
from datetime import datetime, timezone
import subprocess
import sys

# Set this to the Drive folder with the NPZ files directly inside it.
TRACES_DIR = ''
SMOKE = True

traces = Path(TRACES_DIR)
if not TRACES_DIR or not any(traces.glob('*.npz')):
    raise ValueError('Set TRACES_DIR to the Drive folder containing the original dev NPZs.')
stamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
mode = 'smoke' if SMOKE else 'full'
output = Path('/content/drive/MyDrive/rank-kv-followup') / f'{mode}-{stamp}'
cmd = [sys.executable, '-u', 'validation/rerun_longctx_dev.py', str(traces), '--out', str(output)]
if SMOKE:
    cmd.append('--smoke')
with subprocess.Popen(cmd, cwd=project, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
    for line in proc.stdout:
        print(line, end='')
    if proc.wait() != 0:
        raise RuntimeError('Replay failed. See the trace or control mismatch above.')
print((output / 'summary.json').read_text())

A smoke run writes `rank_by_layer.csv` and `summary.json`. The full run also writes `rank_lodo.csv` and `auc.csv`. Compare the original and corrected LODO entries in `summary.json` before updating any poster result. Keep the complete-window AUC column separate from the tie-only comparison.

## All-page rank control (dev only)

The archived ablation `ScoreEMA(signal="rank")` gives every unselected page 0, so it removes both score magnitude **and** the ordering among unselected pages. Its drop in G (0.279 to 0.016 at B=96) therefore cannot tell the two apart. `signal="allrank"` keeps each page's percentile rank among all eligible pages at the step (mid-rank ties) and discards only magnitude.

This cell replays z, allrank and selected-only rank with update scope `all`, the ema(z,all) alpha grid and the same leave-one-document-out alpha selection, at B = 72, 80, 96, 128 and 256. It needs the same 14 dev NPZs and a bundle that contains `validation/rank_control_dev.py` (build the zip from the `review-fixes/ep-rank-control` branch or later). Every replayed LRU, Bélády, `ema:*` and `ema_rank:*` row is checked against the archived dev CSVs, so a different trace file stops the run.

Run the two cells above first (they unpack the bundle and set `TRACES_DIR`; `SMOKE = True` there is enough). Start with `RC_SMOKE = True` (layer 0 of one document), then set it to `False`. The full run is about 32 configs x 5 budgets x 28 layers x 14 documents; use a high-CPU runtime and set `WORKERS` to its core count. Results are not part of the frozen held-out evaluation.


In [ ]:
import os

RC_SMOKE = True
WORKERS = os.cpu_count() or 2

if not (project / 'validation/rank_control_dev.py').is_file():
    raise FileNotFoundError('This bundle predates the all-page rank control; rebuild rank-followup-colab.zip.')
stamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
rc_output = Path('/content/drive/MyDrive/rank-kv-followup') / f"rank-control-{'smoke' if RC_SMOKE else 'full'}-{stamp}"
cmd = [sys.executable, '-u', 'validation/rank_control_dev.py', str(traces), '--out', str(rc_output),
       '--workers', str(WORKERS)]
if RC_SMOKE:
    cmd.append('--smoke')
with subprocess.Popen(cmd, cwd=project, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
    for line in proc.stdout:
        print(line, end='')
    if proc.wait() != 0:
        raise RuntimeError('Rank control failed. See the trace or archive mismatch above.')
print((rc_output / 'summary.json').read_text())


The full run writes `misses.csv`, `lodo.csv`, `summary.csv`, `contrast.csv` and `summary.json`. Read the B=96 rows of `summary.csv` and the `allrank-z` / `allrank-rank` rows of `contrast.csv`: allrank close to z means the ordering among unselected pages carries most of the signal; allrank close to selected-only rank means magnitude matters. G here is relative to uniform-budget LRU and Bélády at the same B. Copy the output folder into `docs/results/longctx_dev/rank_control/` before changing any claim in the docs.
